# Project starter - Airfoil noise

Read [the project questions](project1.md) before starting. Run the setup cells, then fill in the blank code and Markdown cells. The setup prepares the data; it does **not** fit the models or answer the questions for you.


## Setup

You need NumPy, pandas, SciPy, scikit-learn, and Matplotlib. If an import below fails, run `%pip install numpy pandas scipy scikit-learn matplotlib` in a new notebook cell, then restart the kernel. No manual implementation of a solver or statistical test is required. **Do not change the seed, split code, fixed model settings, or reporting threshold below**; they make the numerical outputs comparable for grading.

**Required format for grading:** Use the exact `Q1a`–`Q3d` output labels specified below. Print every named scalar on its own labeled line (show at least four decimal places for computed probabilities, fractions, and errors), and display tables with the exact column names requested. Calculate results in code; do not type expected numbers into print statements. For every confusion matrix, class 0 means pass and class 1 means high noise; rows are actual classes and columns are predicted classes. Use the matching conventional meanings of TN, FP, FN, and TP in summary tables. Put written explanations in the Markdown answer cells, include the requested labeled figures, and use test results only at the end of each task: Question 2(d) for regression and Question 3(d) for classification.

For example, `print(f"Q1a observed_pass_fraction: {value:.4f}")` shows the expected print style; replace `value` with your computed variable. Place the label on the line immediately before each requested table.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, SGDRegressor, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, confusion_matrix, accuracy_score

PROJECT_SEED = 65000
np.random.seed(PROJECT_SEED)
SGD_SETTINGS = {
    "loss": "squared_error",
    "penalty": None,
    "learning_rate": "adaptive",
    "eta0": 0.001,
    "max_iter": 5000,
    "tol": 1e-6,
    "random_state": PROJECT_SEED + 2,
}
LOGISTIC_SETTINGS = {
    "max_iter": 2000,
    "random_state": PROJECT_SEED + 3,
}
REPORTING_THRESHOLD = 0.30
print(f"Project seed: {PROJECT_SEED}")


## Question 1 - Fixed-condition quality-assurance audit

These are **hypothetical audit counts**, separate from the public UCI dataset. A pass means measured noise below 125 dB. Under fixed conditions, 57 of 60 independent runs passed. The benchmark is 0.90 and the pre-audit prior is Beta(9, 1).


In [ ]:
qa_trials = 60
qa_passes = 57
benchmark = 0.90
significance_level = 0.05
prior_a, prior_b = 9, 1


### 1(a) Frequentist test

Call `stats.binomtest` with the one-sided alternative specified in the project sheet. Print **`Q1a observed_pass_fraction`**, **`Q1a exact_p_value`**, and **`Q1a decision`** (use `Reject H0` or `Fail to reject H0`). Interpret the p-value in the Markdown cell below. Do not write a binomial-test implementation.


**Interpretation of the frequentist result:**



### 1(b) Bayesian update

Use the Beta–Bernoulli update and `stats.beta.sf` (the survival function, `1 - CDF`). Print **`Q1b posterior_parameters`** as `(a, b)`, **`Q1b prior_P_above_0.90`**, and **`Q1b posterior_P_above_0.90`**. Explain the update in the Markdown cell below. No numerical-integration loop is needed.


**Q1b interpretation:**



### 1(c) Bayes factor and prior sensitivity

Using the probabilities from 1(b), print **`Q1c prior_odds_H1_to_H0`**, **`Q1c posterior_odds_H1_to_H0`**, and **`Q1c BF10`**. Then repeat the posterior calculation with a fresh Beta(1, 1) prior; print **`Q1c alternative_posterior_parameters`** as `(a, b)` and **`Q1c alternative_posterior_P_above_0.90`**. Explain the Bayes factor and what changed under the alternative prior in the Markdown cell below.


**Q1c interpretation:**



### 1(d) Operational interpretation

In the Markdown cell below, write four to six sentences comparing the p-value, posterior probability, and Bayes factor. State whether you would claim the pass probability exceeds 90%, and name one audit assumption you would check.


**Q1d audit recommendation:**



## Question 2 - Predict the sound-pressure level

The next cell downloads the [official UCI data file](https://archive.ics.uci.edu/ml/machine-learning-databases/00291/airfoil_self_noise.dat) beside this notebook, then reads it with pandas. Questions 2 and 3 use this same dataset. It is tab-separated and has no header row. If the automatic download fails, download `airfoil_self_noise.dat` manually from [the dataset page](https://archive.ics.uci.edu/dataset/291/airfoil%2Bself-noise) and place it beside this notebook before rerunning the cell.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00291/airfoil_self_noise.dat"
FEATURE_COLUMNS = [
    "frequency_hz",
    "angle_of_attack_deg",
    "chord_length_m",
    "free_stream_velocity_m_s",
    "displacement_thickness_m",
]
TARGET_COLUMN = "sound_pressure_db"
data_path = Path("airfoil_self_noise.dat")
if not data_path.exists():
    urlretrieve(DATA_URL, data_path)
airfoil = pd.read_csv(data_path, sep="\t", header=None, names=FEATURE_COLUMNS + [TARGET_COLUMN])
print("Loaded from:", data_path)
print("Rows and columns:", airfoil.shape)
display(airfoil[FEATURE_COLUMNS].head())


### 2(a) Inspect the data

Print **`Q2a row_count`**, **`Q2a input_columns`** (the five names), **`Q2a target_and_unit`** (name and dB), and **`Q2a missing_by_column`** (a dictionary or one table with a count for every column). Give the input units in the Markdown cell. Wait until **after the split** to inspect the target distribution or make its histogram; keep the test targets held out.


**Data-structure check:**



#### Shared split for Questions 2 and 3

The next cell creates the continuous target `y_db` and binary target `y_high_noise`. It then makes **one** 60%/20%/20% train/validation/test split for both tasks using `PROJECT_SEED`. The measured sound-pressure level is never included in `X`. Keep this split code unchanged and reuse these objects for both Questions 2 and 3; do not resplit separately for classification.


In [ ]:
NOISE_LIMIT_DB = 125.0  # illustrative design screen, not a universal standard
X = airfoil[FEATURE_COLUMNS].copy()
y_db = airfoil[TARGET_COLUMN].copy()
y_high_noise = (y_db >= NOISE_LIMIT_DB).astype(int)  # 1 = high noise

X_dev, X_test, y_db_dev, y_db_test, y_high_dev, y_high_test = train_test_split(
    X, y_db, y_high_noise, test_size=0.20, random_state=PROJECT_SEED
)
X_train, X_val, y_db_train, y_db_val, y_high_train, y_high_val = train_test_split(
    X_dev, y_db_dev, y_high_dev, test_size=0.25, random_state=PROJECT_SEED + 1
)
print("Train / validation / test rows:", len(X_train), len(X_val), len(X_test))


#### Training-target distribution

Using `y_db_train` only, print **`Q2a train_min_dB`**, **`Q2a train_median_dB`**, and **`Q2a train_max_dB`**. Display one histogram titled **Training-set sound-pressure levels**, with a dB-labeled horizontal axis. Do not use the validation or test targets for this exploratory plot.


### 2(b) Direct least-squares regression

Call `make_pipeline(StandardScaler(), LinearRegression())`. Fit on `X_train` and `y_db_train` only. Print **`Q2b validation_MAE_dB`** and **`Q2b validation_RMSE_dB`** from validation predictions. This is the direct numerical least-squares fit; do not form a matrix inverse.


### 2(c) Gradient-based regression

Call `make_pipeline(StandardScaler(), SGDRegressor(**SGD_SETTINGS))` on the **same** training data. Keep `SGD_SETTINGS` unchanged. Print **`Q2c SGD_settings`** (the provided dictionary), **`Q2c SGD_epochs_used`**, **`Q2c validation_MAE_dB`**, and **`Q2c validation_RMSE_dB`**. If a convergence warning appears, report it without changing the graded settings. Do not implement SGD yourself.


### 2(d) Compare the regression models

Print **`Q2d validation_regression`** and display a two-row table with columns **`model`**, **`MAE_dB`**, and **`RMSE_dB`**. Display one measured-versus-predicted scatter plot for both models titled **Validation predictions**, with both axes in dB. Print **`Q2d chosen_regression_model`** and justify it using validation results. **After choosing**, print **`Q2d test_regression`** and display a two-row test table with the same columns. You may compute RMSE as `np.sqrt(mean_squared_error(y_true, y_pred))`.


**Regression interpretation:** Which fit would you report, and how similar are the predictions? What might a straight-line model miss?



## Question 3 - Predict high noise

The positive class is `1 = measured noise at or above 125 dB`. Use the existing `X_train`, `X_val`, `X_test` and matching `y_high_*` labels. Do **not** give the measured dB target to the classifier as an input.



### 3(a) Check class balance

Print **`Q3a class_balance`** and display a two-row table (training, validation) with columns **`split`**, **`rows`**, **`high_noise_count`**, and **`high_noise_fraction`**. Save classification test results for Question 3(d). Explain in a Markdown answer why this is now a binary classification task.


**Q3a interpretation:** Explain why predicting whether the measured level crosses 125 dB is binary classification, while predicting the level itself is regression.



### 3(b) Fit logistic regression and inspect probabilities

Call `make_pipeline(StandardScaler(), LogisticRegression(**LOGISTIC_SETTINGS))`, fitting only on training rows. Keep `LOGISTIC_SETTINGS` unchanged. Obtain validation probabilities with `predict_proba`. Print **`Q3b probability_class_order`** (the order of classes in `predict_proba`), **`Q3b regularization`** (penalty and `C`), and **`Q3b validation_examples`**. For the last output, display the **first five validation rows** in a table with columns **`measured_dB`**, **`true_high_noise`**, and **`predicted_P_high_noise`**. Explain what the displayed probabilities mean in the Markdown cell below.


**Q3b interpretation:**



### 3(c) Choose a decision threshold on validation data

Print **`Q3c validation_thresholds`** and display a three-row table for the logistic model (thresholds 0.3, 0.5, 0.7) with columns **`threshold`**, **`TN`**, **`FP`**, **`FN`**, **`TP`**, and **`accuracy`**. Print **`Q3c confusion_matrix_at_0.5`** and display its labeled 2×2 matrix. Print **`Q3c reporting_threshold`** using the fixed `REPORTING_THRESHOLD`. In the Markdown cell, explain the false-negative/false-positive tradeoff and whether you would recommend a different threshold in practice. A **false negative** is a truly high-noise case predicted to pass. Do not use classification test results to make this judgment.


**Q3c threshold interpretation:**



### 3(d) Final logistic-model test

Now evaluate the logistic model on the held-out test set using the fixed `REPORTING_THRESHOLD`. Print **`Q3d test_high_noise_count`** and **`Q3d test_high_noise_fraction`**. Print **`Q3d logistic_test`** and display a one-row table with columns **`TN`**, **`FP`**, **`FN`**, **`TP`**, and **`accuracy`**. In the Markdown answer, interpret the missed high-noise cases and unnecessary warnings. Do not use either regression model as a classifier in Question 3.


**Q3d interpretation:** Explain what the logistic model's false negatives and false positives mean for an initial noise-screening tool.

